In [3]:
cd /home/nampv1/projects/vnpost_asr/

/home/nampv1/projects/vnpost_asr


In [5]:
import os

In [4]:
raw_data_dir = "/media/nampv1/hdd/data/ASR-CommonVoice22Vi-21H/raw/cv-corpus-22.0-2025-06-20/vi"

In [21]:
eda_data_dir = "/media/nampv1/hdd/data/ASR-CommonVoice22Vi-21H/eda/"
os.makedirs(eda_data_dir, exist_ok=True)

In [3]:
import os

In [4]:
os.listdir(raw_data_dir)

['clips',
 'clip_durations.tsv',
 'dev.tsv',
 'invalidated.tsv',
 'other.tsv',
 'reported.tsv',
 'test.tsv',
 'train.tsv',
 'unvalidated_sentences.tsv',
 'validated.tsv',
 'validated_sentences.tsv']

In [5]:
len(os.listdir(os.path.join(raw_data_dir, "clips")))

18761

In [9]:
import pandas as pd

In [11]:
df_clip_durations = pd.read_csv(os.path.join(raw_data_dir, "clip_durations.tsv"), sep="\t")

In [77]:
df_clip_durations

,clip,duration[ms]
0,common_voice_vi_25226546.mp3,2196
1,common_voice_vi_30579154.mp3,4248
2,common_voice_vi_24124535.mp3,4716
3,common_voice_vi_24552157.mp3,4176
4,common_voice_vi_24523018.mp3,5256
...,...,...
18756,common_voice_vi_24535282.mp3,4608
18757,common_voice_vi_26227221.mp3,2376
18758,common_voice_vi_25273399.mp3,3636
18759,common_voice_vi_25268008.mp3,4536


In [75]:
def calculate_total_duration(df: pd.DataFrame, col="duration[ms]"):
    """
    Calculate total duration in ms, seconds, minutes, and hours.

    Args:
        df: pandas DataFrame
        col: column name containing duration in milliseconds

    Returns:
        dict with total durations
    """
    total_ms = df[col].sum()
    total_s = total_ms / 1000
    total_min = total_s / 60
    total_hr = total_min / 60
    return {
        "total_ms": total_ms,
        "total_s": total_s,
        "total_min": total_min,
        "total_hr": round(total_hr, 2)
    }

# Example usage:
# durations = calculate_total_duration(df_clip_durations)
# print(durations)


In [76]:
calculate_total_duration(df_clip_durations)

{'total_ms': np.int64(74430080),
 'total_s': np.float64(74430.08),
 'total_min': np.float64(1240.5013333333334),
 'total_hr': np.float64(20.68)}

In [24]:
import pandas as pd

# calculate total duration in hours (rounded to 2 decimals)
total_duration_hr = round(df_clip_durations["duration[ms]"].sum() / 1000 / 60 / 60, 2)

# create dataframe with only the column required
df_info = pd.DataFrame({"total_duration_hr": [total_duration_hr]})
df_info_path = os.path.join(eda_data_dir, "df_info.csv")
# save to csv
df_info.to_csv(df_info_path, index=False)


In [23]:
df_info

,total_duration_hr
0,20.68


In [27]:
import pandas as pd

# load your existing df_info.csv
df_info = pd.read_csv(df_info_path)

# compute additional statistics
df_info["total_clips"] = len(df_clip_durations)
df_info["avg_duration_s"] = df_clip_durations["duration[ms]"].mean() / 1000
df_info["min_duration_s"] = df_clip_durations["duration[ms]"].min() / 1000
df_info["max_duration_s"] = df_clip_durations["duration[ms]"].max() / 1000
df_info["median_duration_s"] = df_clip_durations["duration[ms]"].median() / 1000

# # save updated df_info
# df_info.to_csv("df_info.csv", index=False)


In [28]:
df_info

,total_duration_hr,total_clips,avg_duration_s,min_duration_s,max_duration_s,median_duration_s
0,20.68,18761,3.967277,0.072,96.552,3.78


In [29]:
df_info.to_csv(df_info_path, index=False)

### Clips

In [131]:
len(os.listdir(os.path.join(raw_data_dir, "clips")))

18761

### Train Val Test Splits

In [30]:
def count_samples(split_df: pd.DataFrame):
    """Count samples in a split file."""
    return len(split_df)

def sentence_length_distribution(split_df: pd.DataFrame, col="sentence"):
    """Return stats on sentence lengths (# words, # chars)."""
    split_df["n_words"] = split_df[col].str.split().str.len()
    split_df["n_chars"] = split_df[col].str.len()
    return split_df[["n_words", "n_chars"]].describe()

def speaker_distribution(split_df: pd.DataFrame, col="client_id"):
    """Count clips per speaker."""
    return split_df[col].value_counts()

In [34]:
def eda_pipeline_splits(train_df: pd.DataFrame, dev_df: pd.DataFrame, test_df: pd.DataFrame):
    """
    Run EDA for train/dev/test splits and return a summary dict.
    Assumes dataframes have columns: 'path', 'sentence', 'client_id'
    """
    summary = {}

    for split_name, df in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
        summary[split_name] = {
            "total_samples": count_samples(df),
            "sentence_length_stats": sentence_length_distribution(df).to_dict(),
            "speaker_distribution": speaker_distribution(df).to_dict()
        }

    # check overlaps across splits
    leakage = check_train_dev_test_leakage(train_df, dev_df, test_df, col="path")
    summary["leakage"] = {k: len(v) for k, v in leakage.items()}

    return summary


In [37]:
import pandas as pd

# ---------- Load data ----------
train_df = pd.read_csv(os.path.join(raw_data_dir, "train.tsv"), sep="\t")
dev_df = pd.read_csv(os.path.join(raw_data_dir, "dev.tsv"), sep="\t")
test_df = pd.read_csv(os.path.join(raw_data_dir, "test.tsv"), sep="\t")

# ---------- Helper functions ----------
def count_samples(split_df: pd.DataFrame):
    return len(split_df)

def sentence_length_distribution(split_df: pd.DataFrame, col="sentence"):
    split_df["n_words"] = split_df[col].str.split().str.len()
    split_df["n_chars"] = split_df[col].str.len()
    return split_df[["n_words", "n_chars"]].describe()

def speaker_distribution(split_df: pd.DataFrame, col="client_id"):
    return split_df[col].value_counts()

def check_train_dev_test_leakage(train_df: pd.DataFrame, dev_df: pd.DataFrame, test_df: pd.DataFrame, col="path"):
    train_set, dev_set, test_set = set(train_df[col]), set(dev_df[col]), set(test_df[col])
    return {
        "train_dev_overlap": train_set & dev_set,
        "train_test_overlap": train_set & test_set,
        "dev_test_overlap": dev_set & test_set
    }

# ---------- Pipeline ----------
def eda_pipeline_splits(train_df: pd.DataFrame, dev_df: pd.DataFrame, test_df: pd.DataFrame):
    summary = {}

    for split_name, df in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
        summary[split_name] = {
            "total_samples": count_samples(df),
            "sentence_length_stats": sentence_length_distribution(df).to_dict(),
            "speaker_distribution": speaker_distribution(df).to_dict()
        }

    leakage = check_train_dev_test_leakage(train_df, dev_df, test_df, col="path")
    summary["leakage"] = {k: len(v) for k, v in leakage.items()}

    return summary

# ---------- Run EDA ----------
summary_splits = eda_pipeline_splits(train_df, dev_df, test_df)

# Print summary
import pprint
pprint.pprint(summary_splits)


{'dev': {'sentence_length_stats': {'n_chars': {'25%': 19.0,
                                               '50%': 30.0,
                                               '75%': 40.0,
                                               'count': 931.0,
                                               'max': 66.0,
                                               'mean': 30.793770139634802,
                                               'min': 6.0,
                                               'std': 13.154434383233903},
                                   'n_words': {'25%': 5.0,
                                               '50%': 7.0,
                                               '75%': 10.0,
                                               'count': 931.0,
                                               'max': 14.0,
                                               'mean': 7.46186895810956,
                                               'min': 2.0,
                                               'std': 3

In [40]:
train_df.columns, len(train_df)

(Index(['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain',
        'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant',
        'locale', 'segment', 'n_words', 'n_chars'],
       dtype='object'),
 2104)

In [42]:
from src.utils.utils import save_dict_to_json

In [43]:
save_dict_to_json(summary_splits, os.path.join(eda_data_dir, "summary_splits.json"))

### invalidated.tsv, validated.tsv, reported.tsv

In [60]:
invalidated_df = pd.read_csv(os.path.join(raw_data_dir, "invalidated.tsv"), sep="\t")
invalidated_df.columns

Index(['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain',
       'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant',
       'locale', 'segment'],
      dtype='object')

In [61]:
validated_df = pd.read_csv(os.path.join(raw_data_dir, "validated.tsv"), sep="\t")
validated_df.columns


Index(['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain',
       'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant',
       'locale', 'segment'],
      dtype='object')

In [95]:
reported_df = pd.read_csv(os.path.join(raw_data_dir, "validated.tsv"), sep="\t")
reported_df.columns


Index(['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain',
       'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant',
       'locale', 'segment'],
      dtype='object')

In [96]:
reported_df.head()

,client_id,path,sentence_id,sentence,sentence_domain,up_votes,down_votes,age,gender,accents,variant,locale,segment
0,01bf7c314cf2359f27dc75f3804fa59615559e3a4c6dc6...,common_voice_vi_30580094.mp3,d9dcf2b14ceefa5981a3c0e7fb785acef8e96b4b45a08a...,ạ. Dạ không ạ. Ngại quá,NaN,2,0,NaN,NaN,NaN,NaN,vi,NaN
1,0461667afd31934657343669d1100d51cc9c2fbc1f1881...,common_voice_vi_41512444.mp3,2cff2cadc56e88c93406c76bae293e47439713f76fc074...,Mẹ của Thảo nói,NaN,3,0,NaN,NaN,NaN,NaN,vi,NaN
2,06aaee8c80892e6f6b781d722ae639db56e4909b0f06db...,common_voice_vi_28846908.mp3,fdc1d23eb998ea4f2aba682c626857b747e459ee301e3d...,Hương thu còn thoảng đâu đây bên thềm,NaN,2,0,NaN,NaN,NaN,NaN,vi,NaN
3,0974be83d83aa95122658767b15250444ba2ad40d3f091...,common_voice_vi_42723041.mp3,35e01717a4b804e8aeef326a29b05f494b97f856781c9c...,Anh ta gật đầu,NaN,3,0,twenties,NaN,NaN,Hà Nội,vi,NaN
4,10597e3d9371d95228dbe8896363354a405ddcce8d4fe3...,common_voice_vi_37195495.mp3,124c6b05ee23d07efeac60c2e4ca894303df0ec3adb040...,thì vẫn không hiện tượng gì tiếp tục xảy đến,NaN,2,0,NaN,NaN,NaN,NaN,vi,NaN


In [62]:
invalidated_df.head()

,client_id,path,sentence_id,sentence,sentence_domain,up_votes,down_votes,age,gender,accents,variant,locale,segment
0,89a110b28009cf4c728e334d874ad7bc9cac4b6365a28f...,common_voice_vi_21824031.mp3,16841507c49f66d62d30f99e2a89748e440b74c1f89116...,"Sóng trước đổ đâu, sóng sau đổ đó.",NaN,1,2,thirties,male_masculine,NaN,NaN,vi,NaN
1,89a110b28009cf4c728e334d874ad7bc9cac4b6365a28f...,common_voice_vi_21824032.mp3,0adde998d421f43c0042a230559b0c3698f20209e27224...,mai mày nhớ cho tao đi cùng đấy,NaN,0,2,thirties,male_masculine,NaN,NaN,vi,NaN
2,89a110b28009cf4c728e334d874ad7bc9cac4b6365a28f...,common_voice_vi_21824033.mp3,0257582bcbd33d94bdf3d9958f98195ffb3b44f899db23...,Khiến cho Trinh bứt rứt không yên,NaN,0,2,thirties,male_masculine,NaN,NaN,vi,NaN
3,89a110b28009cf4c728e334d874ad7bc9cac4b6365a28f...,common_voice_vi_21824034.mp3,09931888a5d42baaebd57128f7a7f98efe65f4eff5a2f7...,Tim nghẹt thở linh hồn như hóa đá,NaN,0,2,thirties,male_masculine,NaN,NaN,vi,NaN
4,89a110b28009cf4c728e334d874ad7bc9cac4b6365a28f...,common_voice_vi_21824047.mp3,1a2cfedc73e466873c100280a1a5ac18f74ccba12008a0...,A hội nằm nhìn ra bên ngoài có cánh cửa đại mở,NaN,1,2,thirties,male_masculine,NaN,NaN,vi,NaN


In [63]:
validated_df

,client_id,path,sentence_id,sentence,sentence_domain,up_votes,down_votes,age,gender,accents,variant,locale,segment
0,01bf7c314cf2359f27dc75f3804fa59615559e3a4c6dc6...,common_voice_vi_30580094.mp3,d9dcf2b14ceefa5981a3c0e7fb785acef8e96b4b45a08a...,ạ. Dạ không ạ. Ngại quá,NaN,2,0,NaN,NaN,NaN,NaN,vi,NaN
1,0461667afd31934657343669d1100d51cc9c2fbc1f1881...,common_voice_vi_41512444.mp3,2cff2cadc56e88c93406c76bae293e47439713f76fc074...,Mẹ của Thảo nói,NaN,3,0,NaN,NaN,NaN,NaN,vi,NaN
2,06aaee8c80892e6f6b781d722ae639db56e4909b0f06db...,common_voice_vi_28846908.mp3,fdc1d23eb998ea4f2aba682c626857b747e459ee301e3d...,Hương thu còn thoảng đâu đây bên thềm,NaN,2,0,NaN,NaN,NaN,NaN,vi,NaN
3,0974be83d83aa95122658767b15250444ba2ad40d3f091...,common_voice_vi_42723041.mp3,35e01717a4b804e8aeef326a29b05f494b97f856781c9c...,Anh ta gật đầu,NaN,3,0,twenties,NaN,NaN,Hà Nội,vi,NaN
4,10597e3d9371d95228dbe8896363354a405ddcce8d4fe3...,common_voice_vi_37195495.mp3,124c6b05ee23d07efeac60c2e4ca894303df0ec3adb040...,thì vẫn không hiện tượng gì tiếp tục xảy đến,NaN,2,0,NaN,NaN,NaN,NaN,vi,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5386,c14ad590de005be3a512e187900c5ad5c76921cf1883b3...,common_voice_vi_24777518.mp3,ee5127fbce27ab866ca51ee1ff18e7ce08b0e1affc2219...,nỗi ám ảnh,NaN,2,0,sixties,male_masculine,NaN,NaN,vi,NaN
5387,c14ad590de005be3a512e187900c5ad5c76921cf1883b3...,common_voice_vi_24777522.mp3,2bf590fe41de4f08c58f79390158a0a3d7316dc7b60b82...,Trâu già thích gặm cỏ non.,NaN,2,0,sixties,male_masculine,NaN,NaN,vi,NaN
5388,c14ad590de005be3a512e187900c5ad5c76921cf1883b3...,common_voice_vi_24830940.mp3,d7b360fe639ac90cf6285e8416e4ee609d78bd80ef5aaa...,tao thấy ở số nhà tao suốt ngày đánh chửi nhau,NaN,2,0,sixties,male_masculine,NaN,NaN,vi,NaN
5389,c14ad590de005be3a512e187900c5ad5c76921cf1883b3...,common_voice_vi_28066159.mp3,9ea634ff574097fc04a46a8f258360cb1992caf9472e40...,Tháng mười về... Thu rất vội em ơi!,NaN,2,0,sixties,male_masculine,NaN,NaN,vi,NaN


In [54]:
train_df.head()

,client_id,path,sentence_id,sentence,sentence_domain,up_votes,down_votes,age,gender,accents,variant,locale,segment,n_words,n_chars
0,c14ad590de005be3a512e187900c5ad5c76921cf1883b3...,common_voice_vi_23901117.mp3,29c4a8e3c89d23ac5090bd6f261bca177782b47dbb7382...,quả nhiên trúng tuyển vào trường Quốc Lập,NaN,2,0,NaN,NaN,NaN,NaN,vi,NaN,8,41
1,c14ad590de005be3a512e187900c5ad5c76921cf1883b3...,common_voice_vi_23901118.mp3,36bafc639499218043c210823a37f69a26b2466ddddbca...,Khi con có mẹ,NaN,2,1,NaN,NaN,NaN,NaN,vi,NaN,4,13
2,c14ad590de005be3a512e187900c5ad5c76921cf1883b3...,common_voice_vi_23901119.mp3,3465b55d25c307976525e8fdf3c267728042cb52092fd7...,nghe Mẹ Thảo nói bố thảo đi làm trưa mới về,NaN,2,0,NaN,NaN,NaN,NaN,vi,NaN,11,43
3,c14ad590de005be3a512e187900c5ad5c76921cf1883b3...,common_voice_vi_23901120.mp3,3825ffe57bb9de4d3d2b76488611e7af7e2a6ababecda9...,Hà ni đánh bạo gọi khẽ,NaN,2,0,NaN,NaN,NaN,NaN,vi,NaN,6,22
4,c14ad590de005be3a512e187900c5ad5c76921cf1883b3...,common_voice_vi_23901121.mp3,12eae4e5371727982f9c38d6e02c4015e28a60db96aade...,Nước chảy trên sông,NaN,2,0,NaN,NaN,NaN,NaN,vi,NaN,4,19


In [58]:
# check overlap between invalidated and train sets
invalidated_paths = set(df_invalidated["path"])
train_paths = set(test_df["path"])

overlap = invalidated_paths & train_paths

print(f"Number of overlaps: {len(overlap)}")
print("Example overlaps:", list(overlap)[:10])


Number of overlaps: 0
Example overlaps: []


### other.tsv

In [68]:
other_df = pd.read_csv(os.path.join(raw_data_dir, "other.tsv"), sep="\t")
other_df.columns

Index(['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain',
       'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant',
       'locale', 'segment'],
      dtype='object')

In [69]:
other_df.head()

,client_id,path,sentence_id,sentence,sentence_domain,up_votes,down_votes,age,gender,accents,variant,locale,segment
0,7606c9e1bbfc6ce41565a075ca737ceeeb5037f5a0407f...,common_voice_vi_23847390.mp3,01a12af2d8eea06d0559636599dab1170263a04f09fca7...,Ni đi theo lúc này cô mới nhanh chóng đóng sầm...,NaN,1,1,teens,male_masculine,NaN,NaN,vi,NaN
1,61a3bdce724f826cd1b2381b019e286fc7dfc0db75b806...,common_voice_vi_23847779.mp3,0deef0cca45773ba99fbd14c5d9fcb2358af479cb23dad...,Đúng rồi mày à,NaN,1,1,NaN,NaN,NaN,NaN,vi,NaN
2,61a3bdce724f826cd1b2381b019e286fc7dfc0db75b806...,common_voice_vi_23847782.mp3,34591b8a0c99fe3fd2f17e3654f405d6a0fecab0d87d0f...,Lòng người hiểm sâu thước nào đo được,NaN,1,1,NaN,NaN,NaN,NaN,vi,NaN
3,61a3bdce724f826cd1b2381b019e286fc7dfc0db75b806...,common_voice_vi_23847783.mp3,1b8836cfb2a62be9c4e2e2221fcce36a776b6422057cd1...,A Hội cầm lấy bao thuốc của tôi,NaN,1,1,NaN,NaN,NaN,NaN,vi,NaN
4,c44e0fee41f88045a040f1ffc8930be3c2a1bcc6f198f5...,common_voice_vi_23848856.mp3,33d915b5ab9b815f62ffca05a97a7ad79f0aa06a27bc86...,Có phải mẹ không ạ,NaN,1,1,twenties,male_masculine,NaN,NaN,vi,NaN


In [71]:
def check_overlap(df1: pd.DataFrame, df2: pd.DataFrame, col="path", sample_n=10):
    """
    Check overlap between two dataframes by column (default: 'path').

    Args:
        df1, df2: pandas DataFrames
        col: column name to compare
        sample_n: number of overlapping examples to return

    Returns:
        dict with overlap count and sample examples
    """
    set1, set2 = set(df1[col]), set(df2[col])
    overlap = set1 & set2
    return {
        "overlap_count": len(overlap),
        "sample_examples": list(overlap)[:sample_n]
    }

In [74]:
# Example usage:
result = check_overlap(other_df, train_df, col="path")
print(result)

{'overlap_count': 0, 'sample_examples': []}


### unvalidated_sentences.tsv, validated_sentences.tsv

In [112]:
unvalidated_sentences_df = pd.read_csv(os.path.join(raw_data_dir, "unvalidated_sentences.tsv"), sep="\t")
unvalidated_sentences_df.columns

Index(['sentence_id', 'sentence', 'sentence_domain', 'source'], dtype='object')

In [113]:
unvalidated_sentences_df

,sentence_id,sentence,sentence_domain,source
0,0022871089ce67bc2b7af856776be81229bf2beb99eee7...,Mà còn là Quỷ Sai của lữu quỷ,NaN,I took sentences from a website of Vietnamese ...
1,00c3942a7e3e994e7549fa759e7416f8549b1362ce9b20...,Lại bia nữa á? kinh nhì,NaN,th? 6 8
2,01653de08bbce9fb30dd85380649e388529f919e2c6935...,sau khi đã đào được lộ hết tất cả chiếc áo qua...,NaN,I took sentences from a website of Vietnamese ...
3,01f03ec6a528cfd8d38706cff500f9e045d1b7b02d9c48...,Ông Khanh hãi hục quá,NaN,I took sentences from a website of Vietnamese ...
4,0201c3bf4f9d37ae2821ecfccd2a8c5168126caa42f110...,Vậy mà những nhát dao tư đâm vào người và vào ...,NaN,I took sentences from a website of Vietnamese ...
...,...,...,...,...
5459,ffd67d563e3e244e1ad337b75854dc78a3b481ed543ef5...,"Biên vô sổ, cuối tháng trả",general,"Từ Điển Tiếng Huế by Bùi Minh Đức, 2001, First..."
5460,ffdf64e053b166b8f28c57dd7e2af6ddadb3cfb1f8573f...,Dù rằng còn mãi xa xăm,NaN,th? l?c bát
5461,ffe498742e03d9cc1be59f72048b6eda340fedf5f555c7...,Mà chạy đến đây nên không cần phải chỉ đường,NaN,I took sentences from a website of Vietnamese ...
5462,ffe5e411d410dd8612338e5bee3aa21f303f6c060db8f6...,Một con mèo đen xì tai dài mõm dài,NaN,I took sentences from a website of Vietnamese ...


In [125]:
validated_sentences_df = pd.read_csv(os.path.join(raw_data_dir, "validated_sentences.tsv"), sep="\t")
validated_sentences_df.columns

Index(['sentence_id', 'sentence', 'sentence_domain', 'source', 'is_used',
       'clips_count'],
      dtype='object')

In [126]:
validated_sentences_df

,sentence_id,sentence,sentence_domain,source,is_used,clips_count
0,01c5579a51cd1a2662b6ed7b3c3d39da10caff8a6fc98b...,Thằng nớ tướng bặm trợn,general,"Từ Điển Tiếng Huế by Bùi Minh Đức, 2001, First...",1,0
1,03f956d92362bd125deda4e65a95b2becb0e6450cde1e5...,Không còn động đất nữa nha,NaN,th? l?c bát,1,0
2,0407f4488b6c7b7c1d0425f41b3c926e2991955c564e74...,rồi trộn chung với gạo nếp giã nhuyễn,NaN,I took sentences from a website of Vietnamese ...,1,0
3,049fadb984bbfdf64bfe99cd88f93e2e25ec51dc48ae74...,"Nghe em đau, đang đêm anh băng đồng băng hói t...",general,"Từ Điển Tiếng Huế by Bùi Minh Đức, 2001, First...",1,0
4,04b3bda8efc196d0e5326d0a0019310c0461335c81b2c5...,Họp hội đông chi mà bát nháo như họp chợ,general,"Từ Điển Tiếng Huế by Bùi Minh Đức, 2001, First...",1,0
...,...,...,...,...,...,...
6270,a7b8f4582757e8cec76fe573bf2c054bbb2a31163368d5...,hôm đấy em chạy qua thấy mọi người xúm lại gần đó,NaN,sentence-collector,1,5
6271,bb07c6b79356ab35a579ef0c61626207e7497904f4ce0b...,Tuy nhiên đó là những điều ai cũng biết,NaN,sentence-collector,1,5
6272,d3e73d6cc91a4c9d7029031303fda6f3c6ad7419c8fc1b...,Đã lâu lắm rồi Trinh mới cảm thấy mình ham muố...,NaN,sentence-collector,1,5
6273,d7a423fdaf94006fd907442c6e7a31448538371c566967...,Ta thấy gì đâu sau sắc yêu kiều,NaN,sentence-collector,1,5


In [115]:
train_df['sentence_id']

0       29c4a8e3c89d23ac5090bd6f261bca177782b47dbb7382...
1       36bafc639499218043c210823a37f69a26b2466ddddbca...
2       3465b55d25c307976525e8fdf3c267728042cb52092fd7...
3       3825ffe57bb9de4d3d2b76488611e7af7e2a6ababecda9...
4       12eae4e5371727982f9c38d6e02c4015e28a60db96aade...
                              ...                        
2099    cca33523a0107fb3534b112f8adf7a817ddbea8177492b...
2100    f30c69e66c98a64a252096f81c344c7864b9f4ee9728d5...
2101    ee5127fbce27ab866ca51ee1ff18e7ce08b0e1affc2219...
2102    d7b360fe639ac90cf6285e8416e4ee609d78bd80ef5aaa...
2103    727c55e4d48b9e40079193e72a0dac0d19879f7c4119b5...
Name: sentence_id, Length: 2104, dtype: object

In [116]:
def check_df_consistency(df1: pd.DataFrame, df2: pd.DataFrame, col1="path", col2="clip", sample_n=10):
    """
    Check consistency of entries between two dataframes based on given columns.

    Args:
        df1, df2: pandas DataFrames to compare
        col1: column in df1
        col2: column in df2
        sample_n: number of sample examples to return

    Returns:
        dict with counts and sample examples of missing entries
    """
    set1, set2 = set(df1[col1]), set(df2[col2])

    missing_in_df2 = set1 - set2
    missing_in_df1 = set2 - set1

    return {
        "missing_in_df2_count": len(missing_in_df2),
        "missing_in_df2_examples": list(missing_in_df2)[:sample_n],
        "missing_in_df1_count": len(missing_in_df1),
        "missing_in_df1_examples": list(missing_in_df1)[:sample_n],
    }



In [129]:
# Example usage:
# compare reported_df with df_clip_durations
report = check_df_consistency(validated_sentences_df, test_df, col1="sentence_id", col2="sentence_id")
report

{'missing_in_df2_count': 5261,
 'missing_in_df2_examples': ['c9c563a36509a9175d46d8557bc48e670757ba89be6080b1e8ea1789e7196d16',
  '81b057cffeb7e1490fc9eb3271ecb56d5b786496a1357f3c0c3b5965360db0d7',
  'c64bb83fb25b1966123ab14cbfb458289a095939496b55bd7268fe6a2e9c7c38',
  '5cf3880ef533bd1a7d00831a74cac79b67b5383ab9a619a6c0f4461b9a0bec42',
  '624ccbb8170cc0d4a6a36e4be4ebf0a156b30ea821181f8600e0d0ed261f4608',
  'f2a7c029a585ef1dfb360f3aab59aa308e3c7dce7be4a761b1c24c5e7337eec9',
  '74a7d91347e66669243275b04ed8ec8a5e18bec9e1739f99e662af36777aeb89',
  'ba0e8fedc970862903da5b81c665226bc92bab64d56108a31e0f85c2c1221f56',
  'aa67aa9ffd7b168f90c3067a7d3d470275ed1d4d6fa3aeba63d21640614e88b5',
  '89ae42592b5769cada2005e56b291cfb093120b1573f8e76c7603491afd03134'],
 'missing_in_df1_count': 0,
 'missing_in_df1_examples': []}

In [123]:
len(unvalidated_sentences_df), len(train_df)

(5464, 2104)

In [119]:
def check_df_consistency(df1: pd.DataFrame, df2: pd.DataFrame, col1="path", col2="clip", sample_n=10):
    set1, set2 = set(df1[col1]), set(df2[col2])

    missing_in_df2 = set1 - set2
    missing_in_df1 = set2 - set1

    return {
        "missing_in_df2_count": len(missing_in_df2),
        "missing_in_df2_examples": list(missing_in_df2)[:sample_n],
        "missing_in_df1_count": len(missing_in_df1),
        "missing_in_df1_examples": list(missing_in_df1)[:sample_n],
    }

def check_all_splits_against_durations(split_dfs: dict, durations_df: pd.DataFrame, col1="path", col2="clip", sample_n=10):
    """
    Loop through multiple split DataFrames and check consistency against durations_df.

    Args:
        split_dfs: dict of {name: DataFrame}
        durations_df: DataFrame containing clip durations
        col1: column name in split DataFrames (default: "path")
        col2: column name in durations_df (default: "clip")
        sample_n: number of sample examples to return

    Returns:
        dict with consistency reports for each split
    """
    results = {}
    for name, df in split_dfs.items():
        results[name] = check_df_consistency(df, durations_df, col1=col1, col2=col2, sample_n=sample_n)
    return results

# Example usage:
split_dfs = {
    "train": train_df,
    "dev": dev_df,
    "test": test_df,
    "invalidated": df_invalidated,
    "reported": reported_df,
    # add validated_df etc. if available
}


In [ ]:

consistency_reports = check_all_splits_against_durations(split_dfs, df_clip_durations)

import pprint
pprint.pprint(consistency_reports)

{'dev': {'missing_in_df1_count': 17830,
         'missing_in_df1_examples': ['common_voice_vi_25723353.mp3',
                                     'common_voice_vi_27658252.mp3',
                                     'common_voice_vi_24566183.mp3',
                                     'common_voice_vi_24570138.mp3',
                                     'common_voice_vi_24121967.mp3',
                                     'common_voice_vi_27973286.mp3',
                                     'common_voice_vi_24398693.mp3',
                                     'common_voice_vi_24173162.mp3',
                                     'common_voice_vi_24304407.mp3',
                                     'common_voice_vi_24255746.mp3'],
         'missing_in_df2_count': 0,
         'missing_in_df2_examples': []},
 'invalidated': {'missing_in_df1_count': 18328,
                 'missing_in_df1_examples': ['common_voice_vi_25723353.mp3',
                                             'common_voice_vi_27658

### Creat HF Dataset Dict

In [6]:
import os
import pandas as pd
from datasets import Dataset, DatasetDict, Audio

def create_hf_dataset(raw_data_dir: str) -> DatasetDict:
    """
    Create a Hugging Face DatasetDict (train/dev/test) with audio + durations.

    Args:
        raw_data_dir (str): Path to dataset folder (with .tsv files + clips/ + clip_durations.tsv)

    Returns:
        DatasetDict: Hugging Face dataset with train, dev, test splits
    """
    # --- Load TSV splits ---
    split_files = {
        "train": "train.tsv",
        "dev": "dev.tsv",
        "test": "test.tsv"
    }
    split_dfs = {
        split: pd.read_csv(os.path.join(raw_data_dir, fname), sep="\t")
        for split, fname in split_files.items()
    }

    # --- Load durations ---
    durations_df = pd.read_csv(os.path.join(raw_data_dir, "clip_durations.tsv"), sep="\t")
    durations_df = durations_df.rename(columns={"clip": "path", "duration[ms]": "duration_ms"})

    # --- Merge + add absolute audio paths ---
    for split, df in split_dfs.items():
        df = df.merge(durations_df, on="path", how="left")  # attach durations
        df["path"] = df["path"].apply(lambda x: os.path.join(raw_data_dir, "clips", x))  # full path
        split_dfs[split] = df

    # --- Convert to HF Datasets ---
    hf_splits = {
        split: Dataset.from_pandas(df, preserve_index=False)
        for split, df in split_dfs.items()
    }

    # --- Add separate audio column ---
    for split, ds in hf_splits.items():
        ds = ds.cast_column("path", Audio())  # decode audio from "path"
        ds = ds.rename_column("path", "audio")  # keep it under "audio"
        hf_splits[split] = ds

    # --- Build DatasetDict ---
    dataset = DatasetDict(hf_splits)
    return dataset


# Example usage:
# raw_data_dir = "/path/to/commonvoice"
# dataset = create_hf_dataset(raw_data_dir)
# print(dataset)
# print(dataset["train"][0])


In [ ]:
create_hf_dataset
dataset = (raw_data_dir)
dataset["train"][0]

{'client_id': 'c14ad590de005be3a512e187900c5ad5c76921cf1883b3349cda8e57e390901f453aae312b3c027b31acc18b7e2f3e857b8f8618ab1eb3a0be7298f8d1c001ce',
 'audio': {'path': '/media/nampv1/hdd/data/ASR-CommonVoice22Vi-21H/raw/cv-corpus-22.0-2025-06-20/vi/clips/common_voice_vi_23901117.mp3',
  'array': array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         1.90132641e-06, 7.18296724e-06, 1.22913943e-05], shape=(286848,)),
  'sampling_rate': 48000},
 'sentence_id': '29c4a8e3c89d23ac5090bd6f261bca177782b47dbb7382f96e6436ba76f4943f',
 'sentence': 'quả nhiên trúng tuyển vào trường Quốc Lập',
 'sentence_domain': None,
 'up_votes': 2,
 'down_votes': 0,
 'age': None,
 'gender': None,
 'accents': None,
 'variant': None,
 'locale': 'vi',
 'segment': None,
 'duration_ms': 5976}

In [8]:
dataset['train']

Dataset({
    features: ['client_id', 'audio', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment', 'duration_ms'],
    num_rows: 2104
})

### draft

In [ ]:
# list of all split dataframes
split_dfs = {
    "train": train_df,
    "dev": dev_df,
    "test": test_df,
    "invalidated": df_invalidated,
    "reported": reported_df,
    "validated": validated_df,
    
}

# run check for each split
consistency_results = {}
for name, df in split_dfs.items():
    consistency_results[name] = check_clip_consistency(df, df_clip_durations)

# print results
import pprint
pprint.pprint(consistency_results)


In [ ]:
train_df.iloc[0]

client_id          c14ad590de005be3a512e187900c5ad5c76921cf1883b3...
path                                    common_voice_vi_23901117.mp3
sentence_id        29c4a8e3c89d23ac5090bd6f261bca177782b47dbb7382...
sentence                   quả nhiên trúng tuyển vào trường Quốc Lập
sentence_domain                                                  NaN
up_votes                                                           2
down_votes                                                         0
age                                                              NaN
gender                                                           NaN
accents                                                          NaN
variant                                                          NaN
locale                                                            vi
segment                                                          NaN
n_words                                                            8
n_chars                           

In [79]:
df_clip_durations

,clip,duration[ms]
0,common_voice_vi_25226546.mp3,2196
1,common_voice_vi_30579154.mp3,4248
2,common_voice_vi_24124535.mp3,4716
3,common_voice_vi_24552157.mp3,4176
4,common_voice_vi_24523018.mp3,5256
...,...,...
18756,common_voice_vi_24535282.mp3,4608
18757,common_voice_vi_26227221.mp3,2376
18758,common_voice_vi_25273399.mp3,3636
18759,common_voice_vi_25268008.mp3,4536


In [87]:
df_clip_durations['clip']

0        common_voice_vi_25226546.mp3
1        common_voice_vi_30579154.mp3
2        common_voice_vi_24124535.mp3
3        common_voice_vi_24552157.mp3
4        common_voice_vi_24523018.mp3
                     ...             
18756    common_voice_vi_24535282.mp3
18757    common_voice_vi_26227221.mp3
18758    common_voice_vi_25273399.mp3
18759    common_voice_vi_25268008.mp3
18760    common_voice_vi_25223919.mp3
Name: clip, Length: 18761, dtype: object

In [97]:
reported_df.iloc[0]['path'] in list(df_clip_durations['clip'])

True

In [103]:
def check_clip_consistency(split_df: pd.DataFrame, durations_df: pd.DataFrame, path_col="path", clip_col="clip", sample_n=10):
    """
    Check consistency of clips between a split file and durations file.

    Args:
        split_df: DataFrame with audio paths (e.g., reported.tsv)
        durations_df: DataFrame with clip durations
        path_col: column name in split_df
        clip_col: column name in durations_df
        sample_n: number of sample examples to return

    Returns:
        dict with counts and sample examples of missing entries
    """
    split_paths = set(split_df[path_col])
    duration_clips = set(durations_df[clip_col])

    missing_in_durations = split_paths - duration_clips
    missing_in_split = duration_clips - split_paths

    return {
        "missing_in_durations_count": len(missing_in_durations),
        "missing_in_durations_examples": list(missing_in_durations)[:sample_n],
        "missing_in_split_count": len(missing_in_split),
        "missing_in_split_examples": list(missing_in_split)[:sample_n],
    }


In [105]:

# Example usage:
consistency_reported = check_clip_consistency(reported_df, df_clip_durations)
consistency_reported

{'missing_in_durations_count': 0,
 'missing_in_durations_examples': [],
 'missing_in_split_count': 13370,
 'missing_in_split_examples': ['common_voice_vi_32079979.mp3',
  'common_voice_vi_27658252.mp3',
  'common_voice_vi_24570138.mp3',
  'common_voice_vi_24566183.mp3',
  'common_voice_vi_27973286.mp3',
  'common_voice_vi_25261275.mp3',
  'common_voice_vi_25092658.mp3',
  'common_voice_vi_26363034.mp3',
  'common_voice_vi_24255746.mp3',
  'common_voice_vi_25273327.mp3']}

In [ ]:
# list of all split dataframes
split_dfs = {
    "train": train_df,
    "dev": dev_df,
    "test": test_df,
    "invalidated": df_invalidated,
    "reported": reported_df,
    "validated": validated_df,
    
}

# run check for each split
consistency_results = {}
for name, df in split_dfs.items():
    consistency_results[name] = check_clip_consistency(df, df_clip_durations)

# print results
import pprint
pprint.pprint(consistency_results)


{'dev': {'missing_in_durations_count': 0,
         'missing_in_durations_examples': [],
         'missing_in_split_count': 17830,
         'missing_in_split_examples': ['common_voice_vi_25723353.mp3',
                                       'common_voice_vi_27658252.mp3',
                                       'common_voice_vi_24566183.mp3',
                                       'common_voice_vi_24570138.mp3',
                                       'common_voice_vi_24121967.mp3',
                                       'common_voice_vi_27973286.mp3',
                                       'common_voice_vi_24398693.mp3',
                                       'common_voice_vi_24173162.mp3',
                                       'common_voice_vi_24304407.mp3',
                                       'common_voice_vi_24255746.mp3']},
 'invalidated': {'missing_in_durations_count': 0,
                 'missing_in_durations_examples': [],
                 'missing_in_split_count': 18328,
    

In [108]:
len(dev_df)

931

In [ ]:

# Example usage:
missing_reported = check_missing_in_durations(reported_df, df_clip_durations)
print(missing_reported)

In [45]:
total = 0

tsv_files = [os.path.join(raw_data_dir, fn) for fn in os.listdir(raw_data_dir) if fn.endswith(".tsv")]

for file in tsv_files:
    print(os.path.basename(file))

    df = pd.read_csv(os.path.join(raw_data_dir, file), sep="\t")
    total += len(df)
    print(len(df))

print(total)

clip_durations.tsv
18761
dev.tsv
931
invalidated.tsv
433
other.tsv
11516
reported.tsv
199
test.tsv
1014
train.tsv
2104
unvalidated_sentences.tsv
5464
validated.tsv
5391
validated_sentences.tsv
6275
52088


In [ ]:
931 + 2104 + 1014 +  + 5391 + 433 

9873